# Live Demo: MoS State + Classical Verification of Quantum Learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mos import MoSState
from mos.sampler import QuantumFourierSampler
from ql.prover import MoSProver
from ql.verifier import MoSVerifier, ParityHypothesis, FourierSparseHypothesis
from experiments.harness.phi import make_single_parity

## 1. MoS state in circuit mode

Pick a small `n`, choose $\varphi$ to be a parity (so $\tilde\varphi = \chi_{s^\star}$ has a single Fourier spike at $s^\star$), sample one $f \sim F_D$, and print the circuit that prepares $|\psi_{U_n,f}\rangle = 2^{-n/2}\sum_x |x, f(x)\rangle$ followed by the Hadamard layer used for QFS.

In [ ]:
n = 4
s_star = 0b0101  # target parity
phi = make_single_parity(n, s_star)
state = MoSState(n=n, phi=phi, seed=42) # type: ignore

f = state.sample_f()
print(f"sampled f(x) for x = 0..{2**n - 1}: {f.tolist()}")

qc = state.circuit_prepare_f(f)
for q in range(n + 1):
    qc.h(q)
qc.measure_all()
print(qc.draw(output='text'))

In [ ]:
sampler = QuantumFourierSampler(state, seed=42)
result = sampler.sample(shots=200, mode='circuit')
print(f"post-selection rate: {result.postselection_rate:.3f}  (expected ~0.5)")
dist = result.empirical_distribution()
for s in np.argsort(-dist)[:8]:
    print(f"  s={s} ({format(s, f'0{n}b')}): empirical={dist[s]:.3f}")
print(f"target s*  = {s_star} ({format(s_star, f'0{n}b')})")

## 2. Run the verification protocol

Honest prover does QFS + heavy-coefficient extraction; classical verifier independently estimates the coefficients and either accepts (with a parity hypothesis) or rejects.

In [ ]:
epsilon, delta = 0.25, 0.1

prover = MoSProver(state, seed=1)
msg = prover.run_protocol(epsilon=epsilon, delta=delta, qfs_mode='statevector')
print(msg.summary())

In [ ]:
verifier = MoSVerifier(state, seed=2)
verdict = verifier.verify_parity(msg, epsilon=epsilon, delta=delta)
print(verdict.summary())

if verdict.accepted:
    h = verdict.hypothesis
    assert isinstance(h, ParityHypothesis)
    print(f"\nrecovered s = {h.s} ({format(h.s, f'0{n}b')});  true s* = {s_star} ({format(s_star, f'0{n}b')})")
    print(f"match: {h.s == s_star}")

## 3. Noisy single parity ($\eta = 0.1$)

Same target parity $s^\star = 101$, but each label is independently flipped with probability $\eta = 0.1$ before state preparation. The Fourier spike is damped from $1$ to $(1-2\eta) = 0.8$, so the distribution-class promise becomes $a^2 = b^2 = (1-2\eta)^2 = 0.64$.

We first build the MoS state, sample one $f \sim F_D$ (with noise), and print the QFS circuit; then check the empirical QFS distribution; then run the full protocol.

In [ ]:
eta = 0.3
state_noisy = MoSState(n=n, phi=phi, noise_rate=eta, seed=43)  # type: ignore

f_noisy = state_noisy.sample_f()
print(f"sampled f(x) (with noise): {f_noisy.tolist()}")
print(f"noiseless parity s*·x:     {[bin(s_star & x).count('1') % 2 for x in range(2**n)]}")

qc_noisy = state_noisy.circuit_prepare_f(f_noisy)
for q in range(n + 1):
    qc_noisy.h(q)
qc_noisy.measure_all()
print(qc_noisy.draw(output='text'))

In [ ]:
sampler_noisy = QuantumFourierSampler(state_noisy, seed=42)
result_noisy = sampler_noisy.sample(shots=200, mode='circuit')
print(f"post-selection rate: {result_noisy.postselection_rate:.3f}")
dist_noisy = result_noisy.empirical_distribution()
for s in np.argsort(-dist_noisy)[:8]:
    print(f"  s={s} ({format(s, f'0{n}b')}): empirical={dist_noisy[s]:.3f}")
print(f"target s* = {s_star} ({format(s_star, f'0{n}b')})  (peak survives but is diluted by noise)")

In [ ]:
counts_noisy = np.zeros(2**n, dtype=int)
for bitstring, c in result_noisy.postselected_counts.items():
    counts_noisy[int(bitstring, 2)] = c

labels = [format(s, f'0{n}b') for s in range(2**n)]
colors = ['tab:red' if s == s_star else 'tab:gray' for s in range(2**n)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, counts_noisy, color=colors)
ax.set_xlabel(f"measured string s (n={n} bits)")
ax.set_ylabel("post-selected sample count")
ax.set_title(f"Section 3: noisy single parity (η={eta}), target s* = {format(s_star, f'0{n}b')}")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
a_sq_noisy = (1 - 2 * eta) ** 2

prover_noisy = MoSProver(state_noisy, seed=1)
msg_noisy = prover_noisy.run_protocol(
    epsilon=epsilon, delta=delta, qfs_mode='statevector', classical_samples=20000
)
print(msg_noisy.summary())

verifier_noisy = MoSVerifier(state_noisy, seed=2)
verdict_noisy = verifier_noisy.verify_parity(
    msg_noisy, epsilon=epsilon, delta=delta,
    a_sq=a_sq_noisy, b_sq=a_sq_noisy, num_samples=20000,
)
print(verdict_noisy.summary())

if verdict_noisy.accepted:
    h = verdict_noisy.hypothesis
    assert isinstance(h, ParityHypothesis)
    print(f"\nrecovered s = {h.s} ({format(h.s, f'0{n}b')});  true s* = {s_star} ({format(s_star, f'0{n}b')})")
    print(f"match: {h.s == s_star}  (est coeff ≈ {h.estimated_coefficient:+.3f}, expected ≈ {1 - 2*eta:+.3f})")

## 4. Noisy Fourier-2-sparse function ($\eta = 0.05$)

Build $\tilde\varphi(x) = \tfrac{1}{2}\chi_{s_1}(x) + \tfrac{1}{2}\chi_{s_2}(x)$ with two target parities $s_1 = 011$ and $s_2 = 110$, but each label is independently flipped with probability $\eta = 0.05$ before state preparation. Both Fourier spikes are damped from $0.5$ to $0.5(1-2\eta) = 0.45$, so the distribution-class promise becomes $a^2 = b^2 = (0.5(1-2\eta))^2 = 0.2025$.

As before we build the MoS state, sample one $f$, print the QFS circuit, do a QFS sanity check (now showing **two** damped peaks), then run the verifier in Fourier-sparse mode with $k = 2$.

In [ ]:
s1, s2 = 0b0110, 0b1010
eta_2 = 0.05

def chi(s, x):
    return 1.0 - 2.0 * (bin(s & x).count('1') % 2)

phi_2sparse = [(1.0 - 0.5 * chi(s1, x) - 0.5 * chi(s2, x)) / 2.0 for x in range(2**n)]
state_2 = MoSState(n=n, phi=phi_2sparse, noise_rate=eta_2, seed=42)  # type: ignore

print("exact spectrum:")
for s, c in enumerate(state_2.fourier_spectrum()):
    if abs(c) > 1e-9:
        print(f"  s={s} ({format(s, f'0{n}b')}): {c:+.3f}")

f_2 = state_2.sample_f()
print(f"\nsampled f(x): {f_2.tolist()}")

qc_2 = state_2.circuit_prepare_f(f_2)
for q in range(n + 1):
    qc_2.h(q)
qc_2.measure_all()
print(qc_2.draw(output='text'))

In [ ]:
sampler_2 = QuantumFourierSampler(state_2, seed=42)
result_2 = sampler_2.sample(shots=400, mode='circuit')
print(f"post-selection rate: {result_2.postselection_rate:.3f}")
dist_2 = result_2.empirical_distribution()
for s in np.argsort(-dist_2)[:4]:
    print(f"  s={s} ({format(s, f'0{n}b')}): empirical={dist_2[s]:.3f}")
print(f"targets s1={s1} ({format(s1, f'0{n}b')}), s2={s2} ({format(s2, f'0{n}b')})  (two roughly equal peaks, diluted by noise)")

In [ ]:
counts_2 = np.zeros(2**n, dtype=int)
for bitstring, c in result_2.postselected_counts.items():
    counts_2[int(bitstring, 2)] = c

labels = [format(s, f'0{n}b') for s in range(2**n)]
targets = {s1, s2}
colors = ['tab:red' if s in targets else 'tab:gray' for s in range(2**n)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, counts_2, color=colors)
ax.set_xlabel(f"measured string s (n={n} bits)")
ax.set_ylabel("post-selected sample count")
ax.set_title(
    f"Section 4: noisy Fourier-2-sparse (η={eta_2}), targets "
    f"s1={format(s1, f'0{n}b')}, s2={format(s2, f'0{n}b')}"
)
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
a_sq_2 = (0.5 * (1 - 2 * eta_2)) ** 2

prover_2 = MoSProver(state_2, seed=1)
msg_2 = prover_2.run_protocol(
    epsilon=epsilon, delta=delta, qfs_mode='statevector', classical_samples=20000
)
print(msg_2.summary())

verifier_2 = MoSVerifier(state_2, seed=2)
verdict_2 = verifier_2.verify_fourier_sparse(
    msg_2, epsilon=epsilon, k=2, delta=delta,
    a_sq=a_sq_2, b_sq=a_sq_2, num_samples=20000,
)
print(verdict_2.summary())

if verdict_2.accepted:
    h = verdict_2.hypothesis
    assert isinstance(h, FourierSparseHypothesis)
    recovered = {format(s, f'0{n}b'): round(float(c), 3) for s, c in h.coefficients.items()}
    print(f"\nrecovered: {recovered}")
    print(f"true s1={s1} ({format(s1, f'0{n}b')}), s2={s2} ({format(s2, f'0{n}b')}), each with expected coeff ≈ {0.5*(1-2*eta_2):+.3f}")